# Reproducing CacheGen (SIGCOMM '24): Figure 9 and Table 1

This notebook reproduces two results from *CacheGen: KV Cache Compression and Streaming for Fast Large Language Model Serving* (Liu et al., SIGCOMM 2024) on Chameleon Cloud:

- **Figure 9**: KV cache size vs. quality for CacheGen and the uniform-quantization baseline.
- **Table 1**: Mistral-7B on LongChat, comparing 8-bit quantization (paper: 622 MB, accuracy 1.00) with CacheGen (paper: 176 MB, 0.98).

It also runs our reimplementation of the token-wise **delta encoding** from §5.2 of the paper ([src/delta_codec.py](src/delta_codec.py)). The authors' released codec does not include it.

**What the notebook does:**
1. Reserves one bare-metal GPU node and a floating IP.
2. Launches `CC-Ubuntu24.04-CUDA` on the node.
3. Installs the authors' conda environment and builds the CUDA arithmetic coder.
4. Runs a 3-document smoke test, then the full experiment, in the background on the node.
5. Downloads the figure, the table, and the raw result JSONs into this Jupyter workspace.

**Before you start:**
- Open this notebook from Chameleon's JupyterHub (for example via Trovi). The `chi` library and SSH keys are configured automatically there.
- Your project needs an allocation with GPU node hours.
- Expect roughly 30 minutes for setup, 30 to 60 minutes for the smoke test, and several hours for the full 50-document run. These are estimates.

**Hardware:** the paper used A40/A100 GPUs. Any NVIDIA GPU with at least 24 GB works here (A100, V100 32GB, A30, RTX 6000). The GPU affects speed, not KV cache size or quality, so it doesn't change the Figure 9 / Table 1 numbers.

## 1. Configuration

Edit these values, then run every cell from top to bottom. Every step is safe to re-run: an existing lease, server, environment or KV cache is reused rather than created again.

> **Pick a node that really has a GPU.** Some GigaIO nodes at CHI@TACC have no GPU, or their hardware inventory is out of date. Use https://discover.chameleoncloud.org to find a node, and check the GPU check in step 4 before running anything long.

In [ ]:
# --- Chameleon resources ---
SITE        = "CHI@UC"
NODE_TYPE   = "gpu_v100"        # alternatives at CHI@UC: "gpu_a100_pcie", "gpu_a100_nvlink", "gpu_rtx_6000"
NODE_NAME   = None              # set to e.g. "P3-GPU-010" to pin one node (NODE_TYPE is then ignored)
LEASE_NAME  = "cachegen-fig9"   # an existing ACTIVE lease with this name is reused
LEASE_HOURS = 24
SERVER_NAME = "cachegen-fig9"
IMAGE       = "CC-Ubuntu24.04-CUDA"   # needs the NVIDIA driver; do not use the vGPU image on bare metal

# --- Code ---
REPO_URL    = "https://github.com/helenxtian/CacheGen-reproduce.git"
REPO_BRANCH = "main"
REPO_DIR    = "~/CacheGen-reproduce"

# --- Experiment ---
DATASET   = "longchat"    # Table 1 uses LongChat
START     = 0
END       = 50            # number of documents for the full run (the paper uses 50 for LongChat)
SMOKE_END = 3             # documents in the smoke test

# Passed as environment variables to scripts/fig9_table1.sh
RUN_ENV = {
    "LONGCHAT_JUDGE":  "string",   # "string" = substring match, no API key; "gpt" needs OPENAI_API_KEY
    "CACHEGEN_LEVELS": "1 2 3",    # CacheGen encoding levels (points on the Fig. 9 curve)
    "QUANT_BITS":      "3 4 8",    # uniform-quantization baselines
    "DELTA_ANCHORS":   "level",    # our delta codec; "level int8" runs both anchor modes, "" skips it
}
SMOKE_ENV = dict(RUN_ENV, CACHEGEN_LEVELS="2", QUANT_BITS="8", RESULTS_DIR="smoke_results")

## 2. Select project and site

Pick your project in the dropdown, then run the next cell.

In [ ]:
import datetime, io, os, shlex, tarfile, time
from IPython.display import Image, Markdown, clear_output, display
from chi import context, lease, server

context.version = "1.0"
context.choose_project()
context.use_site(SITE)

## 3. Reserve a GPU node and launch the instance

The lease asks for **one physical host and one floating IP**. The cell refuses to continue if the lease has no `physical:host` reservation. A lease without one is the most common cause of *"No valid host was found"* when launching.

In [ ]:
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=LEASE_HOURS))
if NODE_NAME:
    l.add_node_reservation(amount=1, node_name=NODE_NAME)
else:
    l.add_node_reservation(amount=1, node_type=NODE_TYPE)
l.add_fip_reservation(amount=1)
l.submit(wait_for_active=True, idempotent=True, show="text")   # reuses a non-terminated lease with this name
l.refresh()

if l.status != "ACTIVE":
    raise RuntimeError(f"Lease {LEASE_NAME} is {l.status}. If it is ERROR, the node is probably taken: "
                       "delete the lease, change NODE_TYPE/NODE_NAME, and re-run.")
if not l.node_reservations:
    raise RuntimeError(f"Lease {LEASE_NAME} has no physical:host reservation (only floating IPs). "
                       "Delete it in the GUI and re-run this cell.")
reservation_id = l.node_reservations[0]["id"]
print("Lease:", l.name, l.id)
print("Host reservation:", reservation_id)
print("Ends:", l.end_date, "UTC")

In [ ]:
existing = [s_ for s_ in server.list_servers() if s_.name == SERVER_NAME]
if existing:
    print(f"Reusing server {SERVER_NAME} (status {existing[0].status})")
else:
    server.Server(SERVER_NAME, reservation_id=reservation_id, image_name=IMAGE).submit(
        wait_for_active=True, show="text")    # bare metal takes about 10-20 minutes

s = server.get_server(SERVER_NAME)
if s.status == "ERROR":
    raise RuntimeError("The server is in ERROR state (often 'No valid host was found'). Run "
                       "server.delete_server(s.id), wait a few minutes, and re-run this cell.")

fip = s.get_floating_ip()
if not fip:
    reserved = l.get_reserved_floating_ips()
    fip = server.associate_floating_ip(s.id, reserved[0] if reserved else None)
    s = server.get_server(SERVER_NAME)
s.check_connectivity(show="text")
print(f"Server is up at {fip}. From this Jupyter terminal: ssh cc@{fip}")

Two small helpers to run commands on the node and copy files back. Output streams into the cell.

In [ ]:
def run(cmd, hide=False, warn=False):
    """Run a shell command on the node (as user cc)."""
    with s.ssh_connection() as c:
        return c.run(cmd, hide=hide, warn=warn, pty=False, in_stream=False)


def put_text(text, remote_path):
    with s.ssh_connection() as c:
        c.put(io.BytesIO(text.encode()), remote_path)


def get(remote_path, local_path):
    with s.ssh_connection() as c:
        c.get(remote_path, local_path)

## 4. Check the GPU

Stop here if this fails. A bare-metal node without a visible NVIDIA GPU can't run the experiment, and no driver install will change that.

In [ ]:
r = run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", warn=True, hide=True)
if not r.ok or not r.stdout.strip():
    run("lspci | grep -iE 'nvidia|vga|3d' || true")
    raise RuntimeError("No NVIDIA GPU/driver on this node. If lspci shows no NVIDIA device, the node has no GPU: "
                       "delete the server and lease and reserve a different node.")
print(r.stdout)
run("df -h ~ | tail -1")   # the full run needs about 70 GB for raw KV caches plus about 15 GB for the model

## 5. Install the software (about 20-30 minutes, the first time only)

This runs the README's install steps:
1. Install Miniconda.
2. Clone the repository.
3. Create the `cachegen` conda environment from `env.yaml`. It pins torch 2.3.1, transformers 4.42.3 and fschat 0.2.36, and includes the CUDA 12 compiler.
4. Install LMCache.
5. Build the `torchac_cuda` arithmetic-coding kernel.

In [ ]:
SETUP = f"""
set -e
export CONDA_PLUGINS_AUTO_ACCEPT_TOS=yes
if [ ! -x ~/miniconda3/bin/conda ]; then
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O ~/miniconda.sh
    bash ~/miniconda.sh -b -p ~/miniconda3
fi
source ~/miniconda3/etc/profile.d/conda.sh
for ch in https://repo.anaconda.com/pkgs/main https://repo.anaconda.com/pkgs/r; do
    conda tos accept --override-channels --channel $ch >/dev/null 2>&1 || true
done

if [ ! -d {REPO_DIR} ]; then
    git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else
    git -C {REPO_DIR} pull --ff-only
fi
cd {REPO_DIR}
git log -1 --format='code version: %h %s'

if ! conda env list | grep -q '^cachegen '; then
    conda env create -f env.yaml
fi
conda activate cachegen
pip install -q -e LMCache
if ! python -c 'import torchac_cuda' 2>/dev/null; then
    (cd LMCache/third_party/torchac_cuda && python setup.py install)
fi
python -c 'import torch, torchac_cuda; print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))'
"""
put_text(SETUP, "setup_cachegen.sh")
run("bash setup_cachegen.sh")

## 6. Run the experiment in the background

`scripts/fig9_table1.sh` does five steps:
1. Prefill each document and save its raw fp16 KV cache.
2. CacheGen at encoding levels 1-3.
3. Our delta-encoding codec.
4. Uniform quantization at 3, 4 and 8 bits.
5. The uncompressed baseline.

Each step writes a JSON file to the results folder.

Jobs run detached on the node, so closing this notebook or losing the connection doesn't stop them. `wait_job` only watches the log. If you interrupt it, the job keeps going; re-run the cell to resume watching.

In [ ]:
def launch(tag, start, end, env):
    if run("pgrep -f '[s]cripts/fig9_table1.sh'", warn=True, hide=True).ok:
        print("A fig9_table1.sh job is already running; not starting another. Use wait_job() to watch it.")
        return
    exports = " ".join(f"{k}={shlex.quote(str(v))}" for k, v in env.items())
    # All-or-nothing chain (a failed conda activate must not run the script in the wrong env);
    # its output, including setup errors, goes to the log, and its exit code to <tag>.done
    inner = (f"( source ~/miniconda3/etc/profile.d/conda.sh && conda activate cachegen && cd {REPO_DIR} && "
             f"export CUDA_VISIBLE_DEVICES=0 {exports} && "
             f"bash scripts/fig9_table1.sh {DATASET} {start} {end} ) > ~/logs/{tag}.log 2>&1; "
             f"echo $? > ~/logs/{tag}.done")
    run(f"mkdir -p ~/logs && rm -f ~/logs/{tag}.done && "
        f"nohup setsid bash -c {shlex.quote(inner)} > /dev/null 2>&1 < /dev/null &")
    print(f"Started '{tag}' (docs {start}-{end}). Log on the node: ~/logs/{tag}.log")


def wait_job(tag, poll=60, tail=20):
    t0 = time.time()
    while True:
        try:
            done = run(f"cat ~/logs/{tag}.done 2>/dev/null", warn=True, hide=True)
            log = run(f"tail -n {tail} ~/logs/{tag}.log 2>/dev/null", warn=True, hide=True).stdout
        except Exception as e:      # transient SSH hiccup: retry
            print("ssh error, retrying:", e); time.sleep(poll); continue
        clear_output(wait=True)
        print(f"[{tag}] watching for {(time.time() - t0) / 60:.0f} min\n" + log)
        if done.ok and done.stdout.strip():
            code_ = int(done.stdout.strip())
            print(f"[{tag}] finished with exit code {code_}")
            if code_ != 0:
                raise RuntimeError(f"Job {tag} failed; see the full log with run('less ~/logs/{tag}.log') "
                                   f"or run('tail -n 200 ~/logs/{tag}.log')")
            return
        time.sleep(poll)

### 6a. Smoke test (3 documents, level 2 / 8-bit only)

This checks the whole pipeline before you spend hours on the full run: model download, prefill, the CUDA coder, the delta codec, and evaluation. Results go to `smoke_results/` so they don't mix with the real ones.

In [ ]:
launch("smoke", START, SMOKE_END, SMOKE_ENV)
wait_job("smoke")
run(f"cd {REPO_DIR} && ~/miniconda3/envs/cachegen/bin/python plot_fig9.py --results_dir smoke_results --out smoke_fig9.png")

**What to check before the full run:**
- Every method has a row in the table above.
- CacheGen level 2 is several times smaller than `quant 8bit`. The paper reports about 3.5x.
- Search the log for `estimate` and `clamped` with `run("grep -iE 'estimate|clamped|_mb' ~/logs/smoke.log | head -40")`. The delta codec's clamped fraction should be near 0.

### 6b. Full run

Raw KV caches from the smoke test are reused, so those 3 documents aren't prefilled again.

In [ ]:
launch("full", START, END, RUN_ENV)
wait_job("full", poll=300)

## 7. Collect results

This draws Figure 9 and builds Table 1 on the node, then copies them, the per-method JSONs and the logs into `results_<site>/` in this Jupyter workspace. Files under `~/work` persist after the lease ends.

In [ ]:
RESULTS_DIR = "mistral7b_results"
run(f"cd {REPO_DIR} && PY=~/miniconda3/envs/cachegen/bin/python && "
    f"$PY plot_fig9.py --results_dir {RESULTS_DIR} --out fig9.png && "
    f"$PY make_table1.py --results_dir {RESULTS_DIR} --dataset {DATASET}")
run(f"cd {REPO_DIR} && tar czf ~/cachegen_results.tgz fig9.png {RESULTS_DIR} smoke_results smoke_fig9.png "
    f"-C ~ logs setup_cachegen.sh 2>/dev/null || true")

local_dir = f"results_{SITE.replace('@', '_')}"
os.makedirs(local_dir, exist_ok=True)
get("cachegen_results.tgz", f"{local_dir}/cachegen_results.tgz")
with tarfile.open(f"{local_dir}/cachegen_results.tgz") as t:
    t.extractall(local_dir)
print("Saved to", os.path.abspath(local_dir))

display(Image(f"{local_dir}/fig9.png"))
table_md = f"{local_dir}/{RESULTS_DIR}/table1_{DATASET}.md"
if os.path.exists(table_md):
    display(Markdown(open(table_md).read()))

### Notes for the report

- **Judge:** LongChat accuracy uses a substring match (`LONGCHAT_JUDGE=string`) by default. The paper's artifact used a GPT judge, which you can enable with `LONGCHAT_JUDGE=gpt` and an `OPENAI_API_KEY`.
- **Sizes:**
  - CacheGen sizes are the length of the encoded bitstream.
  - Quantization sizes are computed from the bit width.
  - The fp16 baseline is the size of the raw KV tensor.
- **Not reproduced:** the H2O and LLMLingua rows of Table 1 aren't part of the authors' artifact, so they aren't run here.
- **Delta encoding** (`cachegen_delta`) is our reimplementation of §5.2, not the authors' code. See the docstring in `src/delta_codec.py` for the design choices.
- **Record your hardware:** the GPU model printed in step 4 and the code version printed in step 5.

## 8. Clean up

Delete the server and the lease when you're done so they stop using your allocation. **Download the results first (step 7).** Everything on the node is lost when it's deleted.

In [ ]:
CONFIRM_DELETE = False   # set to True and re-run to delete the server and the lease

if CONFIRM_DELETE:
    server.delete_server(s.id)
    l.delete()
    print("Deleted server and lease.")
else:
    print("Nothing deleted. Set CONFIRM_DELETE = True to release the node.")